In [1]:
import os
import pandas as pd
import numpy as np
from nilearn.image import load_img
from nilearn.masking import compute_epi_mask
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
# ------------ CONFIG ------------
sub = 'sub-01'



base_dir = '/Users/liuq13/SPCD_data'

beta_dir = os.path.join(base_dir, 'GLM', 'odor_scan_trialwise_2mm', sub)
tsv_dir = os.path.join(base_dir, 'nii', sub, 'func_PAID', 'odor_scan')




In [3]:
mask_img = load_img('/Users/liuq13/SPCD_data/masks/gm_0.1.nii')
mask_data = mask_img.get_fdata() > 0  # make it boolean

In [4]:
def load_data_for_run(run):
    tsv_file = os.path.join(tsv_dir, f'{sub}_task-odor_run-{run}_events.tsv')
    events = pd.read_csv(tsv_file, sep='\t')
    events = events[events['trial_type'].isin(['odor_1', 'odor_2'])].copy()
    events['label'] = events['trial_type'].map({'odor_1': 0, 'odor_2': 1})
    events['trial'] = np.arange(1, len(events) + 1)

    X = []

    for trial_num in events['trial']:
        beta_file = os.path.join(beta_dir, f'beta_run{run}_trial{trial_num:02d}.nii')
        img = load_img(beta_file)
        beta_data = img.get_fdata()
        X.append(beta_data[mask_data])

    X = np.vstack(X)

    return X, events['label'].values


In [5]:
# First load the data using the final_mask
X1_raw, y1 = load_data_for_run(1)
X2_raw, y2 = load_data_for_run(2)

# Stack together to find shared valid voxels (no NaNs across either run)
X_stack = np.vstack([X1_raw, X2_raw])
valid_voxels = np.all(np.isfinite(X_stack), axis=0)

# Apply consistent voxel filtering to both runs
X1 = X1_raw[:, valid_voxels]
X2 = X2_raw[:, valid_voxels]


In [7]:
# ------------ Cross-run decoding ------------
clf = SVC(kernel='linear', C=1)

# Train on Run 1, test on Run 2
clf.fit(X1, y1)
y2_pred = clf.predict(X2)
acc1to2 = accuracy_score(y2, y2_pred)

# Train on Run 2, test on Run 1
clf.fit(X2, y2)
y1_pred = clf.predict(X1)
acc2to1 = accuracy_score(y1, y1_pred)

# ------------ Results ------------
print(f'\n=== Cross-Run Decoding: {sub} ===')
print(f'Train on Run 1 → Test on Run 2: {acc1to2:.3f}')
print(f'Train on Run 2 → Test on Run 1: {acc2to1:.3f}')
print(f'Mean cross-run accuracy: {(acc1to2 + acc2to1)/2:.3f}')


=== Cross-Run Decoding: sub-01 ===
Train on Run 1 → Test on Run 2: 0.400
Train on Run 2 → Test on Run 1: 0.550
Mean cross-run accuracy: 0.475
